# 📩 SMS Spam Sınıflandırma — Multinomial Naive Bayes

**🇹🇷 Türkçe:**
Bu notebook, klasik **SMS Spam Collection** veri setinde bir mesajın *spam* mi yoksa *ham* (normal) mı olduğunu tahmin ediyor. Kullanılan model **Multinomial Naive Bayes**; metin, `CountVectorizer` ile "kelime torbası" (Bag of Words) temsiline çevriliyor. Akış: veriyi yükleme → kolon temizliği → kopya satırların kaldırılması → etiket kodlama → tabakalı eğitim/test ayrımı → vektörleştirme → model eğitimi → değerlendirme.

**Neden Multinomial, neden Gaussian değil?** `GaussianNB` her özelliğin sınıf içinde normal dağıldığını varsayar; bu, Iris'teki gibi sürekli ölçümler için uygundur. Burada özelliklerimiz **kelime sayımları** — yani kesikli (discrete), negatif olamayan tam sayılar. `MultinomialNB` tam olarak bu tür sayım verileri için tasarlanmıştır ve metin sınıflandırmanın klasik çözümüdür.

**Neden bu problem zor değil ama öğretici?** Sınıflar **dengesiz** (yaklaşık %87 ham, %13 spam). Bu yüzden doğruluk (accuracy) tek başına yanıltıcıdır: her mesaja "ham" diyen boş bir model bile %87 doğruluk alır. Gerçek başarı, spam sınıfının **recall** değerinde görülür.

**🇬🇧 English:**
This notebook classifies SMS messages as *spam* or *ham* (legitimate) on the classic **SMS Spam Collection** dataset. The model is **Multinomial Naive Bayes**, and the text is converted into a "Bag of Words" representation with `CountVectorizer`. The pipeline: load data → clean columns → drop duplicates → encode labels → stratified train/test split → vectorization → model training → evaluation.

**Why Multinomial and not Gaussian?** `GaussianNB` assumes each feature is normally distributed within a class, which suits continuous measurements like those in Iris. Here the features are **word counts** — discrete, non-negative integers. `MultinomialNB` is designed exactly for such count data and is the classic choice for text classification.

**Why is this problem instructive?** The classes are **imbalanced** (roughly 87% ham, 13% spam), so accuracy alone is misleading: a trivial model that labels everything "ham" would already score 87%. The real signal is the **recall** of the spam class.

---

## 📦 Kütüphaneleri İçe Aktarma / Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 📥 1. Veri Setini Yükleme / Load the Dataset

**🇹🇷 Türkçe:**
Veri seti UTF-8 değil, **`latin-1`** kodlamasıyla saklanmış. `encoding` parametresi verilmezse `pandas` bir `UnicodeDecodeError` fırlatır — eski metin veri setlerinde sık karşılaşılan bir durumdur. Aşağıdaki hücre, dosya yolunu hem Kaggle ortamında hem de yerel repo yapısında çalışacak şekilde otomatik seçiyor.

**🇬🇧 English:**
The dataset is stored in **`latin-1`** encoding rather than UTF-8. Without the `encoding` argument, `pandas` raises a `UnicodeDecodeError` — a common situation with older text datasets. The cell below picks the file path automatically so it works both on Kaggle and in the local repository layout.

In [2]:
import os, glob

# Kaggle'da veri seti /kaggle/input altina baglanir; yerelde repo icindeki CSV kullanilir.
# On Kaggle the dataset is mounted under /kaggle/input; locally the repo CSV is used.
matches = glob.glob('/kaggle/input/**/spam.csv', recursive=True)
path = matches[0] if matches else '../../Data/12-spam.csv'
print('Reading:', path)
df = pd.read_csv(path, encoding='latin-1')


Reading: ../../Data/12-spam.csv


## 🔍 2. İlk Bakış / First Look

**🇹🇷 Türkçe:**
Veri seti beklenenden dağınık geliyor: kolon adları anlamsız (`v1`, `v2`) ve `Unnamed: 2/3/4` adında üç fazladan kolon var. Herhangi bir temizliğe girişmeden önce elimizde tam olarak ne olduğunu görmek gerekiyor.

**🇬🇧 English:**
The dataset arrives messier than expected: the column names are meaningless (`v1`, `v2`) and there are three extra columns named `Unnamed: 2/3/4`. Before doing any cleaning, it is worth seeing exactly what we have.

In [3]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [4]:
df.columns

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='str')

## 🏷️ 3. Kolonları Yeniden Adlandırma / Renaming the Columns

**🇹🇷 Türkçe:**
`v1` ve `v2` hiçbir şey anlatmıyor. `v1` etiketi (`ham`/`spam`), `v2` ise mesaj metnini tutuyor; anlamlı isimlere çevriliyor. Burada yeni kolonlar oluşturulup eskileri siliniyor.

> 💡 Aynı işi tek satırda yapmanın kısa yolu: `df = df.rename(columns={"v1": "categories", "v2": "messages"})`. Sonuç birebir aynı, sadece daha az adım.

**🇬🇧 English:**
`v1` and `v2` convey nothing. `v1` holds the label (`ham`/`spam`) and `v2` the message text, so they are given meaningful names. Here new columns are created and the old ones dropped afterwards.

> 💡 The shorter equivalent in one line: `df = df.rename(columns={"v1": "categories", "v2": "messages"})`. The result is identical, just fewer steps.

In [5]:
df["categories"] = df["v1"]

In [6]:
df["messages"] = df["v2"]

In [7]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4,categories,messages
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN,ham,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
df = df.drop(["v1","v2"], axis=1)

In [9]:
df

,Unnamed: 2,Unnamed: 3,Unnamed: 4,categories,messages
0,NaN,NaN,NaN,ham,"Go until jurong point, crazy.. Available only ..."
1,NaN,NaN,NaN,ham,Ok lar... Joking wif u oni...
2,NaN,NaN,NaN,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,NaN,NaN,NaN,ham,U dun say so early hor... U c already then say...
4,NaN,NaN,NaN,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...,...,...,...
5567,NaN,NaN,NaN,spam,This is the 2nd time we have tried 2 contact u...
5568,NaN,NaN,NaN,ham,Will Ì_ b going to esplanade fr home?
5569,NaN,NaN,NaN,ham,"Pity, * was in mood for that. So...any other s..."
5570,NaN,NaN,NaN,ham,The guy did some bitching but I acted like i'd...


## 🧹 4. Fazla Kolonların İncelenmesi ve Kaldırılması / Inspecting and Dropping the Extra Columns

**🇹🇷 Türkçe:**
`Unnamed: 2/3/4` kolonlarını körü körüne silmek yerine önce içlerine bakıyoruz. Görülen şu: 5572 satırın neredeyse tamamı `NaN`, dolu olan birkaç hücre ise orijinal CSV'de tırnak/virgül hatası yüzünden taşmış mesaj parçaları. Yani gerçek bir bilgi taşımıyorlar ve güvenle kaldırılabilirler.

> 💡 Bir kolonu silmeden önce `unique()` veya `value_counts(dropna=False)` ile içine bakmak iyi bir alışkanlıktır — bazen "boş sanılan" kolonda değerli bilgi çıkar.

**🇬🇧 English:**
Instead of blindly deleting the `Unnamed: 2/3/4` columns, we inspect them first. The result: almost all of the 5,572 rows are `NaN`, and the few filled cells are fragments of messages that spilled over due to quoting/comma issues in the original CSV. They carry no real information and can safely be dropped.

> 💡 Inspecting a column with `unique()` or `value_counts(dropna=False)` before deleting it is a good habit — a column assumed to be empty sometimes hides useful information.

In [10]:
df["Unnamed: 2"].unique()

<StringArray>
[                                                                                                                                               nan,
                                                                                                                                     ' PO Box 5249',
                                                                     ' the person is definitely special for u..... But if the person is so special',
                                                                                         ' HOWU DOIN? FOUNDURSELF A JOBYET SAUSAGE?LOVE JEN XXX\""',
                                                                                             ' wanted to say hi. HI!!!\" Stop? Send STOP to 62468"',
                                                                                                  'this wont even start........ Datz confidence.."',
                                                                                            

In [11]:
df = df.drop(["Unnamed: 2","Unnamed: 3","Unnamed: 4"], axis=1)

In [12]:
df

,categories,messages
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


## ♻️ 5. Kopya Satırların Kaldırılması / Removing Duplicate Rows

**🇹🇷 Türkçe:**
Veri setinde **403 kopya satır** var (%7'ye yakın). Metin sınıflandırmada bu ciddi bir problemdir: aynı mesaj hem eğitim hem test kümesine düşerse, model onu gerçekten genellemiş olmadığı hâlde doğru bilir ve test skoru yapay olarak yükselir. Temizlik sonrası 5572 satır **5169**'a iniyor ve `duplicated().sum()` 0 döndürerek işlemi doğruluyor.

**🇬🇧 English:**
The dataset contains **403 duplicate rows** (nearly 7%). In text classification this matters: if the same message lands in both the training and the test set, the model gets it right by memorization rather than generalization, inflating the test score. After cleaning, 5,572 rows drop to **5,169**, and `duplicated().sum()` returning 0 confirms the operation.

In [13]:
df.duplicated().sum()

np.int64(403)

In [14]:
df = df.drop_duplicates()

In [15]:
df.duplicated().sum()

np.int64(0)

## 🔢 6. Etiketi Sayısala Çevirme / Encoding the Label

**🇹🇷 Türkçe:**
Model metin etiket kabul etmez, bu yüzden `ham` → **0**, `spam` → **1** eşlemesi yapılıyor. Sadece iki sınıf olduğu için burada `LabelEncoder` yerine `map()` tercih edildi: hangi sınıfın 1 olacağına açıkça sen karar verirsin. Bu önemlidir, çünkü ilgilendiğimiz "pozitif" sınıf spam'dir ve `classification_report` çıktısında `1` satırı doğrudan spam performansını gösterecek. Ardından artık gereksiz olan `categories` kolonu kaldırılıyor.

**🇬🇧 English:**
The model cannot consume text labels, so `ham` → **0** and `spam` → **1**. With only two classes, `map()` is preferable to `LabelEncoder` here: you explicitly decide which class becomes 1. That matters because spam is the "positive" class we care about, and row `1` of the `classification_report` will therefore report spam performance directly. The now-redundant `categories` column is then dropped.

In [16]:
df["label"] = df["categories"].map({"ham": 0, "spam": 1})

In [17]:
df

,categories,messages,label
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1
5568,ham,Will Ì_ b going to esplanade fr home?,0
5569,ham,"Pity, * was in mood for that. So...any other s...",0
5570,ham,The guy did some bitching but I acted like i'd...,0


In [18]:
df = df.drop("categories", axis = 1)

## ✂️ 7. Özellik ve Hedef / Feature and Target

**🇹🇷 Türkçe:**
Burada `X`, Iris'teki gibi çok kolonlu bir DataFrame değil; **tek bir metin Series'i**. Sayısal özellikler henüz yok — onları bir sonraki adımlarda `CountVectorizer` üretecek.

**🇬🇧 English:**
Unlike in Iris, `X` here is not a multi-column DataFrame but a **single text Series**. There are no numeric features yet — `CountVectorizer` will generate them in a later step.

In [19]:
X = df["messages"]
y = df["label"]

## 🔀 8. Eğitim / Test Ayrımı — `stratify` / Train-Test Split with `stratify`

**🇹🇷 Türkçe:**
Veri %80 eğitim (4135 satır) ve %20 test (1034 satır) olarak ayrılıyor. Buradaki kritik parametre **`stratify=y`**: sınıf oranlarının hem eğitim hem test kümesinde korunmasını garanti eder (tabakalı örnekleme).

Neden gerekli? Spam oranı yalnızca **%12.63** ve test kümesinde sadece ~131 spam mesajı var. `stratify` olmadan bu sayı `random_state`'e göre 135 ile 149 arasında zıplar; o zaman ölçtüğün skor farkı modelin kalitesinden değil, bölmenin şansından kaynaklanır. `stratify=y` ile test kümesindeki spam oranı her çalıştırmada %12.67'de sabit kalır.

**🇬🇧 English:**
The data is split into 80% training (4,135 rows) and 20% test (1,034 rows). The critical argument here is **`stratify=y`**, which preserves the class proportions in both sets (stratified sampling).

Why it matters: spam accounts for only **12.63%** of the data, leaving roughly 131 spam messages in the test set. Without `stratify`, that count swings between 135 and 149 depending on `random_state`, so differences in your score reflect the luck of the split rather than the quality of the model. With `stratify=y`, the test-set spam ratio stays fixed at 12.67% across runs.

> 💡 Dengesiz veri setlerinde sınıflandırma yaparken `stratify` neredeyse her zaman doğru tercihtir. / For classification on imbalanced data, `stratify` is almost always the right choice.

In [20]:
from sklearn.model_selection import train_test_split

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0, stratify = y)

## 🧪 9. `CountVectorizer` Nasıl Çalışır? / How `CountVectorizer` Works

**🇹🇷 Türkçe:**
Modeller metni değil sayıyı anlar. `CountVectorizer`, her mesajı içindeki kelimelerin **sayımlarına** indirger — kelime sırasını tamamen yok sayar. Bu temsile "kelime torbası" (Bag of Words) denir.

Aşağıdaki üç cümlelik oyuncak örnek fikri net gösteriyor:

| | call | free | me | money | now | will | win | you |
|---|---|---|---|---|---|---|---|---|
| `win free money now` | 0 | 1 | 0 | 1 | 1 | 0 | 1 | 0 |
| `call me now free`   | 1 | 1 | 1 | 0 | 1 | 0 | 0 | 0 |
| `I will call you`    | 1 | 0 | 0 | 0 | 0 | 1 | 0 | 1 |

Her **satır** bir mesaj, her **sütun** bir kelime, hücreler ise sayımlar. Naive Bayes'in öğrendiği şey tam olarak budur: `free` ve `win` sütunları spam satırlarında kabarır, normal mesajlarda sıfır kalır.

Dikkat: sözlükte `"I"` yok. Çünkü `CountVectorizer` varsayılan olarak metni küçük harfe çevirir (`lowercase=True`) ve en az 2 karakterli kelimeleri alır, tek harfleri ve noktalamayı atar.

**🇬🇧 English:**
Models understand numbers, not text. `CountVectorizer` reduces each message to the **counts** of the words it contains, discarding word order entirely. This representation is called "Bag of Words".

The three-sentence toy example below makes the idea concrete: each **row** is a message, each **column** a word, and the cells are counts. This is precisely what Naive Bayes learns from — columns like `free` and `win` spike in spam rows and stay at zero in legitimate ones.

Note that `"I"` is missing from the vocabulary: by default `CountVectorizer` lowercases the text (`lowercase=True`) and keeps only tokens of at least two characters, discarding single letters and punctuation.

In [22]:
from sklearn.feature_extraction.text import CountVectorizer

demo_docs = ['win free money now', 'call me now free', 'I will call you']
demo_cv = CountVectorizer()
demo_matrix = demo_cv.fit_transform(demo_docs)

pd.DataFrame(demo_matrix.toarray(),
             columns=demo_cv.get_feature_names_out(),
             index=['SMS 1', 'SMS 2', 'SMS 3'])


,call,free,me,money,now,will,win,you
SMS 1,0,1,0,1,1,0,1,0
SMS 2,1,1,1,0,1,0,0,0
SMS 3,1,0,0,0,0,1,0,1


## 🔤 10. Gerçek Veriyi Vektörleştirme / Vectorizing the Real Data

**🇹🇷 Türkçe:**
Buradaki en kritik kural, `StandardScaler`'daki ile aynıdır: **`fit_transform` sadece eğitim verisine, `transform` test verisine** uygulanır. Sözlük yalnızca eğitim kümesinden kurulmalıdır; aksi hâlde test setindeki kelime dağarcığı modele sızar (*data leakage*).

Sonuç: 4135 × **7596** boyutunda bir matris — yani 2 kolonluk veri seti bir anda 7596 özelliğe dönüşüyor. Hücrelerin **%99.8'i sıfırdır** (ortalama bir SMS ~15 kelimedir, kalan 7581 sütun boştur). Bu yüzden çıktı normal bir NumPy dizisi değil, seyrek matristir (`csr_matrix`); yalnızca sıfır olmayan değerleri saklayarak bellekten tasarruf eder.

> ⚠️ Bu matrise `.toarray()` uygulamayın — 31 milyon hücreyi belleğe açar. `MultinomialNB` seyrek matrisi doğrudan kabul eder.

Test kümesinde sözlükte bulunmayan yeni bir kelime varsa `transform` onu sessizce yok sayar. Bu doğru davranıştır: gerçek hayatta da model daha önce hiç görmediği kelimelerle karşılaşır.

**🇬🇧 English:**
The critical rule here is the same as with `StandardScaler`: **`fit_transform` on the training data only, `transform` on the test data.** The vocabulary must be built from the training set alone; otherwise the test set's vocabulary leaks into the model (*data leakage*).

The result is a 4,135 × **7,596** matrix — a two-column dataset suddenly has 7,596 features. **99.8% of the cells are zero** (an average SMS has ~15 words, leaving 7,581 empty columns), which is why the output is a sparse matrix (`csr_matrix`) rather than a dense NumPy array: it stores only the non-zero entries.

> ⚠️ Do not call `.toarray()` on this matrix — it would expand 31 million cells into memory. `MultinomialNB` accepts sparse input directly.

If the test set contains a word absent from the vocabulary, `transform` silently ignores it. That is the correct behaviour: in production the model will always meet words it has never seen.

In [23]:
cv = CountVectorizer()

X_train_vec = cv.fit_transform(X_train)   # sozluk SADECE train'den kurulur / vocabulary from train only
X_test_vec = cv.transform(X_test)         # test ayni sozluge gore cevrilir / test mapped to same vocabulary

print('X_train_vec:', X_train_vec.shape)
print('X_test_vec :', X_test_vec.shape)


X_train_vec: (4135, 7596)
X_test_vec : (1034, 7596)


## 🤖 11. Modeli Eğitme / Training the Model — `MultinomialNB`

**🇹🇷 Türkçe:**
`MultinomialNB` her kelime için "spam mesajlarında ne sıklıkta geçiyor / normal mesajlarda ne sıklıkta geçiyor" olasılıklarını öğreniyor, ayrıca sınıfların ön (prior) olasılıklarını hesaplıyor. Tahmin sırasında Bayes Teoremi ile mesajdaki tüm kelimelerin kanıtını birleştirip en yüksek olasılıklı sınıfı seçiyor.

> ℹ️ `MultinomialNB` bir **model**dir, dönüştürücü (transformer) değil — bu yüzden `fit` ve `predict` metotları vardır, `fit_transform` yoktur. `CountVectorizer` ise dönüştürücüdür ve `fit`/`transform` metotlarına sahiptir. Scikit-learn'deki bu ayrım tüm kütüphane boyunca tutarlıdır.

**🇬🇧 English:**
`MultinomialNB` learns, for every word, how frequently it appears in spam versus legitimate messages, along with the class priors. At prediction time it combines the evidence from all words in a message via Bayes' Theorem and picks the most probable class.

> ℹ️ `MultinomialNB` is an **estimator**, not a transformer — hence it exposes `fit` and `predict`, but no `fit_transform`. `CountVectorizer`, being a transformer, exposes `fit`/`transform`. This distinction is consistent throughout scikit-learn.

In [24]:
from sklearn.naive_bayes import MultinomialNB

In [25]:
mnb = MultinomialNB()

In [26]:
mnb.fit(X_train_vec, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3613., 522.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.13,-2.07]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 7596)","[[ 0., 0., 1.,..., 0., 8., 1.], [ 5.,23., 0.,..., 1., 0., 0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 7596)","[[-10.91,-10.91,-10.22,...,-10.91, -8.72,-10.22], [ -8.09, -6.71, -9.88,..., -9.19, -9.88, -9.88]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7596


## 🔮 12. Tahmin / Making Predictions

**🇹🇷 Türkçe:**
Model, daha önce hiç görmediği 1034 test mesajı üzerinde tahmin yapıyor. `y_pred` her mesaj için 0 (ham) veya 1 (spam) değerini içeriyor.

**🇬🇧 English:**
The model predicts on the 1,034 unseen test messages. `y_pred` holds 0 (ham) or 1 (spam) for each message.

In [27]:
y_pred = mnb.predict(X_test_vec)

## 📈 13. Değerlendirme / Evaluation

**🇹🇷 Türkçe:**
Dengesiz veri setinde doğruluk tek başına yeterli değildir, bu yüzden karışıklık matrisi ve `classification_report` birlikte okunuyor.

**Karışıklık matrisi:**

|  | Tahmin: ham | Tahmin: spam |
|---|---|---|
| **Gerçek: ham** | 900 ✅ | 3 ❌ (yanlış alarm) |
| **Gerçek: spam** | 10 ❌ (kaçan spam) | 121 ✅ |

**Sonuçlar:** Doğruluk **%98.74**. Ama asıl önemli olan spam (sınıf 1) satırıdır: **precision 0.98**, **recall 0.92**.

- **Precision 0.98** → Spam dediklerimizin %98'i gerçekten spam. Sadece **3 normal mesaj** yanlışlıkla spam kutusuna düştü.
- **Recall 0.92** → Gerçek spam'lerin %92'sini yakaladık; **10 spam** gözden kaçtı.

Bir spam filtresinde bu ikisi arasında bir denge (trade-off) vardır ve **precision genellikle daha kritiktir**: kullanıcının önemli bir mesajının spam kutusuna düşmesi, birkaç spam'in gelen kutusunda görünmesinden çok daha rahatsız edicidir. Bu açıdan 3 yanlış alarm iyi bir sonuçtur.

Karşılaştırma için: her mesaja "ham" diyen boş bir model %87.3 doğruluk alırdı ama spam recall'ı 0 olurdu. %98.74'ün gerçekten anlamlı olmasının sebebi budur.

**🇬🇧 English:**
On an imbalanced dataset accuracy alone is insufficient, so the confusion matrix and the `classification_report` are read together.

**Confusion matrix:** 900 legitimate messages correctly kept, 3 wrongly flagged as spam; 121 spam messages caught, 10 missed.

**Results:** accuracy **98.74%**. The row that matters, though, is spam (class 1): **precision 0.98**, **recall 0.92**.

- **Precision 0.98** → 98% of the messages we flagged as spam really were spam; only **3 legitimate messages** were misfiled.
- **Recall 0.92** → we caught 92% of the actual spam; **10 messages** slipped through.

A spam filter always trades these off, and **precision is usually the more critical of the two**: having an important message land in the spam folder annoys a user far more than a few spam messages reaching the inbox. By that standard, 3 false alarms is a good outcome.

For comparison, a trivial model labelling everything "ham" would reach 87.3% accuracy with a spam recall of 0. That is why 98.74% here is genuinely meaningful.

In [28]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [29]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(accuracy_score(y_test, y_pred))

[[900   3]
 [ 10 121]]
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       903
           1       0.98      0.92      0.95       131

    accuracy                           0.99      1034
   macro avg       0.98      0.96      0.97      1034
weighted avg       0.99      0.99      0.99      1034

0.9874274661508704


## 🏁 Sonuç / Conclusion

**🇹🇷 Türkçe:**
Multinomial Naive Bayes, basitliğine rağmen SMS spam tespitinde **%98.74 doğruluk** ve **0.95 F1** (spam sınıfı) elde etti. Model saniyeler içinde eğitiliyor ve ayarlanacak neredeyse hiç hiperparametresi yok — metin sınıflandırmada hâlâ güçlü bir taban çizgisi (baseline) olmasının sebebi bu.

"Naive" (saf) bağımsızlık varsayımının burada açıkça yanlış olduğunu belirtmek gerekir: doğal dilde kelimeler kesinlikle birbirinden bağımsız değildir. Buna rağmen model iyi çalışıyor, çünkü sınıflandırma için olasılıkların tam olarak doğru olması gerekmez — doğru sınıfın en yüksek olasılığı alması yeterlidir.

**Denenebilecek sonraki adımlar:**
- `TfidfVectorizer` ile `CountVectorizer`'ı karşılaştırmak (nadir ve ayırt edici kelimeleri ağırlıklandırır).
- `stop_words='english'` ve `min_df=2` parametrelerinin skora etkisini ölçmek.
- `ngram_range=(1, 2)` ile kelime ikililerini ("free entry" gibi) eklemek.
- `predict_proba()` ile spam olasılığını alıp 0.5 eşiğini kaydırarak precision/recall dengesini ayarlamak.
- `cross_val_score` ile çapraz doğrulama yapıp sonucun kararlılığını sınamak.

**🇬🇧 English:**
Despite its simplicity, Multinomial Naive Bayes reached **98.74% accuracy** and an **F1 of 0.95** on the spam class. It trains in seconds and has virtually no hyperparameters to tune, which is why it remains a strong baseline for text classification.

It is worth noting that the "naive" independence assumption is plainly false here: words in natural language are certainly not independent of one another. The model works anyway, because classification does not require the probabilities to be exactly right — it only requires the correct class to come out on top.

**Possible next steps:**
- Compare `TfidfVectorizer` against `CountVectorizer` (it weights rare, discriminative words more heavily).
- Measure the effect of `stop_words='english'` and `min_df=2` on the score.
- Add word pairs such as "free entry" via `ngram_range=(1, 2)`.
- Use `predict_proba()` and shift the 0.5 threshold to tune the precision/recall balance.
- Run `cross_val_score` to check how stable the result is.